In [ ]:
import duckdb
import requests
import pandas as pd

con = duckdb.connect("gdelt.db")

def process_file(url):
    con.execute(f"""
        INSERT INTO weekly_agg
        SELECT
          date_trunc('week', strptime(SQLDATE, '%Y%m%d')) AS week,
          ActionGeo_CountryCode AS country,
          COUNT(*) AS events,
          AVG(AvgTone) AS avg_tone
        FROM read_csv_auto('{url}')
        GROUP BY 1, 2
    """)

In [ ]:
import requests

url = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
text = requests.get(url).text

lines = [l for l in text.splitlines() if "export" in l]

# last 1000
for line in lines[:50]:
    print(line)

150383 297a16b493de7cf6ca809a7cc31d0b93 http://data.gdeltproject.org/gdeltv2/20150218230000.export.CSV.zip
149211 2a91041d7e72b0fc6a629e2ff867b240 http://data.gdeltproject.org/gdeltv2/20150218231500.export.CSV.zip
149723 12268e821823aae2da90882621feda18 http://data.gdeltproject.org/gdeltv2/20150218233000.export.CSV.zip
158842 a5298ce3c6df1a8a759c61b5c0b6f8bb http://data.gdeltproject.org/gdeltv2/20150218234500.export.CSV.zip
362610 c4268d558bb22c02b3c132c17818c68b http://data.gdeltproject.org/gdeltv2/20150219000000.export.CSV.zip
251605 7685a6c71f010918f3be0d4ed2be977e http://data.gdeltproject.org/gdeltv2/20150219001500.export.CSV.zip
255259 f41066efb05d4024fca9dc1c2c6b9112 http://data.gdeltproject.org/gdeltv2/20150219003000.export.CSV.zip
219398 555d808779fe5b3eaf0a9ebf212116a2 http://data.gdeltproject.org/gdeltv2/20150219004500.export.CSV.zip
225092 6b4e1d0421548dbba59754d0f164d2a1 http://data.gdeltproject.org/gdeltv2/20150219010000.export.CSV.zip
185226 36f14471b716d8b47c8f766507ab9a

In [ ]:
len(lines)

386422

In [ ]:
import requests

url = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
text = requests.get(url).text

urls = []
for line in text.splitlines():
    if "gkg" in line:
        parts = line.split()
        urls.append(parts[2])  # third column

# last 10 files
for u in urls[-10:]:
    print(u)

http://data.gdeltproject.org/gdeltv2/20260513030000.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513031500.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513033000.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513034500.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513040000.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513041500.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513043000.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513044500.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513050000.gkg.csv.zip
http://data.gdeltproject.org/gdeltv2/20260513051500.gkg.csv.zip


In [ ]:
for url in urls[:10]:
    process_file(url)

CatalogException: Catalog Error: Table with name weekly_agg does not exist!
Did you mean "pg_am"?

In [ ]:
con = duckdb.connect()

url = "http://data.gdeltproject.org/gdeltv2/20170301100000.gkg.csv.zip"

df = con.execute(f"""
    SELECT
        column0  AS GLOBALEVENTID,
        column1  AS SQLDATE,
        column30 AS GoldsteinScale,
        column34 AS AvgTone,
        column51 AS ActionGeo_CountryCode
    FROM read_csv(
        '{url}',
        delim='\t',
        header=false,
        quote='',
        escape='',
        ignore_errors=true
    )
    LIMIT 5
""").fetchdf()

print(df)

InvalidInputException: Invalid Input Error: Error when sniffing file "http://data.gdeltproject.org/gdeltv2/20170301100000.gkg.csv.zip".
It was not possible to automatically detect the CSV parsing dialect
The search space used was:
Delimiter Candidates: '	'
Quote/Escape Candidates: ['(no quote)','(no escape)']
Comment Candidates: '\0', '#'
Encoding: utf-8
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Delimiter is set to '	'. Consider unsetting it.
* Quote is set to '\0'. Consider unsetting it.
* Escape is set to '\0'. Consider unsetting it.
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable null padding (null_padding=true) to pad missing columns with NULL values
* Check you are using the correct file compression, otherwise set it (e.g., compressi

In [ ]:
import zipfile
import io

In [ ]:
from datetime import datetime, timedelta

# starting timestamp
start = datetime.strptime("20170301000000", "%Y%m%d%H%M%S")

# one full day = 96 intervals of 15 minutes
for i in range(96):
    ts = start + timedelta(minutes=15 * i)
    timestamp = ts.strftime("%Y%m%d%H%M%S")
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.export.CSV.zip"
    print(f"Reading {url}")
    try:
        # download zip
        r = requests.get(url, timeout=30)

        # skip missing files
        if r.status_code != 200:
            print(f"Missing: {url}")
            continue

        # open zip in memory
        z = zipfile.ZipFile(io.BytesIO(r.content))

        # get contained csv filename
        csv_name = z.namelist()[0]

        # read dataframe
        with z.open(csv_name) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                low_memory=False
            )

        print(df.head())
        print(df.shape)
    except Exception as e:
        print(f"Error on {timestamp}: {e}")

Reading http://data.gdeltproject.org/gdeltv2/20170301000000.export.CSV.zip


          0         1       2     3          4    5    6    7    8    9   ...  \
0  632226002  20160301  201603  2016  2016.1671  NaN  NaN  NaN  NaN  NaN  ...   
1  632226003  20160301  201603  2016  2016.1671  NaN  NaN  NaN  NaN  NaN  ...   
2  632226004  20160301  201603  2016  2016.1671  NaN  NaN  NaN  NaN  NaN  ...   
3  632226005  20160301  201603  2016  2016.1671  NaN  NaN  NaN  NaN  NaN  ...   
4  632226006  20160301  201603  2016  2016.1671  NaN  NaN  NaN  NaN  NaN  ...   

  51                                               52  53    54     55  \
0  3             Hollywood, California, United States  US  USCA  CA037   
1  1                                         Colombia  CO    CO    NaN   
2  1                                         Colombia  CO    CO    NaN   
3  1                                             Iraq  IZ    IZ    NaN   
4  3  Washington, District of Columbia, United States  US  USDC  DC001   

        56        57       58              59  \
0  34.0983 -118.327

In [ ]:
def urls_for_day(date_str, kind="export"):
    start = datetime.strptime(date_str, "%Y-%m-%d")
    return [
        f"http://data.gdeltproject.org/gdeltv2/"
        f"{(start + timedelta(minutes=15*i)).strftime('%Y%m%d%H%M%S')}.{kind}.CSV.zip"
        for i in range(96)
    ]

event_urls = urls_for_day("2017-03-01", "export")

In [ ]:
con = duckdb.connect()

url = "http://data.gdeltproject.org/gdeltv2/20170301000000.gkg.csv.zip"

con.execute(f"""
CREATE TEMP TABLE day_events AS
SELECT
    column00::BIGINT  AS GLOBALEVENTID,
    column01::INTEGER AS SQLDATE,
    column30::DOUBLE  AS GoldsteinScale,
    column34::DOUBLE  AS AvgTone,
    column51          AS ActionGeo_CountryCode
FROM read_csv(
    '{url}',
    delim = '\t',
    header = false,
    columns = {{
        'column00': 'VARCHAR', 'column01': 'VARCHAR', 'column02': 'VARCHAR',
        'column03': 'VARCHAR', 'column04': 'VARCHAR', 'column05': 'VARCHAR',
        'column06': 'VARCHAR', 'column07': 'VARCHAR', 'column08': 'VARCHAR',
        'column09': 'VARCHAR', 'column10': 'VARCHAR', 'column11': 'VARCHAR',
        'column12': 'VARCHAR', 'column13': 'VARCHAR', 'column14': 'VARCHAR',
        'column15': 'VARCHAR', 'column16': 'VARCHAR', 'column17': 'VARCHAR',
        'column18': 'VARCHAR', 'column19': 'VARCHAR', 'column20': 'VARCHAR',
        'column21': 'VARCHAR', 'column22': 'VARCHAR', 'column23': 'VARCHAR',
        'column24': 'VARCHAR', 'column25': 'VARCHAR', 'column26': 'VARCHAR',
        'column27': 'VARCHAR', 'column28': 'VARCHAR', 'column29': 'VARCHAR',
        'column30': 'VARCHAR', 'column31': 'VARCHAR', 'column32': 'VARCHAR',
        'column33': 'VARCHAR', 'column34': 'VARCHAR', 'column35': 'VARCHAR',
        'column36': 'VARCHAR', 'column37': 'VARCHAR', 'column38': 'VARCHAR',
        'column39': 'VARCHAR', 'column40': 'VARCHAR', 'column41': 'VARCHAR',
        'column42': 'VARCHAR', 'column43': 'VARCHAR', 'column44': 'VARCHAR',
        'column45': 'VARCHAR', 'column46': 'VARCHAR', 'column47': 'VARCHAR',
        'column48': 'VARCHAR', 'column49': 'VARCHAR', 'column50': 'VARCHAR',
        'column51': 'VARCHAR', 'column52': 'VARCHAR', 'column53': 'VARCHAR',
        'column54': 'VARCHAR', 'column55': 'VARCHAR', 'column56': 'VARCHAR',
        'column57': 'VARCHAR', 'column58': 'VARCHAR', 'column59': 'VARCHAR',
        'column60': 'VARCHAR'
    }},
    quote = '',
    strict_mode = false,
    escape = '',
    null_padding = true,
    ignore_errors = true
)
""")

InvalidInputException: Invalid Input Error: Error when sniffing file "http://data.gdeltproject.org/gdeltv2/20170301000000.gkg.csv.zip".
It was not possible to automatically detect the CSV parsing dialect
The search space used was:
Delimiter Candidates: '	'
Quote/Escape Candidates: ['(no quote)','(no escape)']
Comment Candidates: '\0', '#'
Encoding: utf-8
Possible fixes:
* Columns are set as: "columns = { 'column00' : 'VARCHAR', 'column01' : 'VARCHAR', 'column02' : 'VARCHAR', 'column03' : 'VARCHAR', 'column04' : 'VARCHAR', 'column05' : 'VARCHAR', 'column06' : 'VARCHAR', 'column07' : 'VARCHAR', 'column08' : 'VARCHAR', 'column09' : 'VARCHAR', 'column10' : 'VARCHAR', 'column11' : 'VARCHAR', 'column12' : 'VARCHAR', 'column13' : 'VARCHAR', 'column14' : 'VARCHAR', 'column15' : 'VARCHAR', 'column16' : 'VARCHAR', 'column17' : 'VARCHAR', 'column18' : 'VARCHAR', 'column19' : 'VARCHAR', 'column20' : 'VARCHAR', 'column21' : 'VARCHAR', 'column22' : 'VARCHAR', 'column23' : 'VARCHAR', 'column24' : 'VA

## Aggregation

In [ ]:
con = duckdb.connect("gdelt.db")

con.execute("""
CREATE OR REPLACE TABLE weekly_agg AS
SELECT
    date_trunc(
        'week',
        strptime(CAST(EventTimeDate AS VARCHAR), '%Y%m%d%H%M%S')
    ) AS week,

    ActionGeo_CountryCode AS country,

    COUNT(*) AS mentions,
    COUNT(DISTINCT GlobalEventID) AS events,

    AVG(GoldsteinScale) AS avg_goldstein,
    AVG(MentionDocTone) AS avg_tone,
    AVG(Confidence) AS avg_confidence,

    SUM(NumMentions) AS total_num_mentions,
    SUM(NumSources) AS total_num_sources,
    SUM(NumArticles) AS total_num_articles

FROM read_parquet(
    'D:/gdelt_data/2016_merged/merged*.csv',
    union_by_name=true
)

WHERE ActionGeo_CountryCode IS NOT NULL
  AND ActionGeo_CountryCode != ''

GROUP BY
    week,
    country

ORDER BY
    week,
    country
""")

# sample: 4.1s
# hochskaliert: 120x so viele files -> 120x
# bei O(n) -> ca. 8 Minuten
# bei O(n**2) -> ca. 16 Stunden

con.execute("""
COPY weekly_agg
TO 'D:/gdelt_data/weekly_country_agg.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [ ]:
con = duckdb.connect("gdelt.db")

con.execute("""
CREATE OR REPLACE TABLE weekly_agg AS
SELECT
    date_trunc(
        'week',
        strptime(CAST(EventTimeDate AS VARCHAR), '%Y%m%d%H%M%S')
    ) AS week,

    ActionGeo_CountryCode AS country,

    COUNT(*) AS mentions,
    COUNT(DISTINCT GlobalEventID) AS events,

    AVG(GoldsteinScale) AS avg_goldstein,
    AVG(MentionDocTone) AS avg_tone,
    AVG(Confidence) AS avg_confidence,

    SUM(NumMentions) AS total_num_mentions,
    SUM(NumSources) AS total_num_sources,
    SUM(NumArticles) AS total_num_articles

FROM read_parquet(
    'D:/gdelt_data/2016_merged/merged*.csv',
    union_by_name=true
)

WHERE ActionGeo_CountryCode IS NOT NULL
  AND ActionGeo_CountryCode != ''

GROUP BY
    week,
    country

ORDER BY
    week,
    country
""")

# sample: 4.1s
# hochskaliert: 120x so viele files -> 120x
# bei O(n) -> ca. 8 Minuten
# bei O(n**2) -> ca. 16 Stunden

con.execute("""
COPY weekly_agg
TO 'D:/gdelt_data/weekly_country_agg.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")